# Hypothesis Testing

Rigorous statistical methods for evaluating mechanistic interpretability claims:
permutation tests, bootstrapped confidence intervals, multiple comparison
correction, effect sizes, and circuit hypothesis testing.

## Why This Matters

Model hypothesis testing provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.hypothesis_testing import (
    permutation_test,
    bootstrap_confidence_interval,
    multiple_comparison_correction,
    effect_size_analysis,
    circuit_hypothesis_test,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

# Two groups of inputs
tokens_a = [jnp.array([1, 2, 3, 4, 5, 6, 7]), jnp.array([2, 3, 4, 5, 6, 7, 8]),
            jnp.array([3, 4, 5, 6, 7, 8, 9])]
tokens_b = [jnp.array([50, 51, 52, 53, 54, 55, 56]), jnp.array([51, 52, 53, 54, 55, 56, 57]),
            jnp.array([52, 53, 54, 55, 56, 57, 58])]

def metric_fn(logits, tokens):
    return jnp.mean(logits[-1])

print("Model and data ready")

## Permutation Test

Test whether two groups differ significantly on a metric.

In [ ]:
result = permutation_test(model, tokens_a, tokens_b, metric_fn, n_permutations=500)
print(f"Observed difference: {result['observed_diff']:.6f}")
print(f"P-value: {result['p_value']:.4f}")
print(f"Effect size (Cohen's d): {result['effect_size']:.4f}")
print(f"Significant at α=0.05? {'Yes' if result['p_value'] < 0.05 else 'No'}")

## Bootstrap Confidence Intervals

In [ ]:
all_tokens = tokens_a + tokens_b
result = bootstrap_confidence_interval(model, all_tokens, metric_fn, n_bootstrap=500)
print(f"Point estimate: {result['point_estimate']:.6f}")
print(f"95% CI: [{result['ci_lower']:.6f}, {result['ci_upper']:.6f}]")
print(f"Standard error: {result['standard_error']:.6f}")

## Multiple Comparison Correction

When testing many hypotheses, correct for multiple comparisons.

In [ ]:
p_values = [0.001, 0.01, 0.03, 0.04, 0.5, 0.8]

for method in ['bonferroni', 'fdr']:
    result = multiple_comparison_correction(p_values, method=method)
    print(f"\n{method.upper()}:")
    for i, (orig, corr, sig) in enumerate(zip(
        p_values, result['corrected_p_values'], result['significant'])):
        print(f"  p={orig:.3f} -> {float(corr):.4f} {'*' if sig else ''}")
    print(f"  Significant: {result['n_significant']}/{len(p_values)}")

## Effect Size Analysis

Measure how much an ablation changes a metric, with confidence intervals.

In [ ]:
import equinox as eqx

def zero_head_ablation(m):
    """Zero out head 0 in layer 0."""
    new_W_O = m.blocks[0].attn.W_O.at[0].set(jnp.zeros_like(m.blocks[0].attn.W_O[0]))
    return eqx.tree_at(lambda x: x.blocks[0].attn.W_O, m, new_W_O)

result = effect_size_analysis(model, all_tokens, metric_fn, zero_head_ablation, n_bootstrap=200)
print(f"Mean effect: {result['mean_effect']:.6f}")
print(f"Cohen's d: {result['cohens_d']:.4f}")
print(f"95% CI: [{result['ci_lower']:.6f}, {result['ci_upper']:.6f}]")

## Circuit Hypothesis Test

Test whether a hypothesized circuit is genuinely important vs random components.

In [ ]:
result = circuit_hypothesis_test(
    model, all_tokens, metric_fn,
    circuit_components=[(0, 0), (1, 0)],
    n_permutations=50,
)
print(f"Circuit effect: {result['circuit_effect']:.6f}")
print(f"P-value: {result['p_value']:.4f}")
print(f"Specificity: {result['specificity']:.4f}")
print(f"Null mean: {float(jnp.mean(result['null_effects'])):.6f}")